---
title: Plotting xarray data from the Open Data Cube
short_title: Plotting
subject: Beginner Guide
subtitle: Visualising xarray Datasets and DataArrays returned by dc.load.
description: Visualising xarray Datasets and DataArrays returned by dc.load.
keywords:
  - open-data-cube
  - odc
  - xarray
  - plotting
  - beginner-guide
---

This notebook explains how to plot xarray data returned by `dc.load`, from a single band to a natural-colour composite and a facet grid of comparable panels.[^edits]

[^edits]: Tutorial notebooks update automatically, so edits may be overwritten during an update.
    Keep a working copy in a separate file to preserve changes.

## A. Objectives

- Plot a single band as a colour-mapped image
- Stack the red, green, and blue bands into a natural-colour composite
- Compare one band across years in a row of panels

## B. Loading sample data

Load a small dataset containing four bands for 2024 and 2025.

In [ ]:
from datacube import Datacube

dc = Datacube(app="plotting")

query = {
    "product": "s2_geomad_annual",
    "x": (98.80, 98.90),
    "y": (2.65, 2.55),
    "time": ("2024", "2025"),
    "measurements": ["red", "green", "blue", "nir"],
    "output_crs": "EPSG:32647",
    "resolution": (-30, 30),
}

ds = dc.load(**query)

## C. Plotting

Plots turn arrays of pixel values into images that can reveal spatial patterns and unusual values.
In a Jupyter notebook, the result appears directly below the code, making it easier to relate each plotting choice to the image it produces.

An image panel needs two spatial dimensions.
The loaded `xarray.Dataset` also contains several bands and years, so the data must first be arranged according to what the plot should show.

## D. Single-band image

Begin with the red band for 2024.
Selecting one band and one year leaves a 2D DataArray with the spatial dimensions `y` and `x`, which xarray can draw as an image.

In [ ]:
ds.red.isel(time=0).plot()

The code chains three operations.
First, `ds.red` selects the red band.
Then `.isel(time=0)` selects the first loaded year by position, which is 2024.
Finally, `.plot()` draws the 2D values and adds a colour bar that relates each colour to a pixel value.

The `cmap` argument controls the colour map used for those values.
Matplotlib provides a range of built-in options in its [colour-map reference](https://matplotlib.org/stable/gallery/color/colormap_reference.html).
A sequential colour map is useful here because it shows lower and higher values along one ordered scale.
For example, `"Reds"` uses shades from pale to dark red:

In [ ]:
ds.red.isel(time=0).plot(cmap="Reds")

## E. Multi-band composite

A natural-colour image assigns the red, green, and blue measurements to the corresponding display channels.
These bands must first be combined into one DataArray in the correct order.

In [ ]:
rgb = ds[["red", "green", "blue"]].to_array(dim="band").isel(time=0)
rgb

The selection `ds[["red", "green", "blue"]]` creates a smaller Dataset containing the three required bands in red, green, blue order.
The `.to_array(dim="band")` call stacks those data variables along a new dimension named `band`.
The `dim` argument is optional; without it, xarray names the new dimension `"variable"`.
Next, `.isel(time=0)` selects the 2024 data.

The result is stored in the Python variable `rgb` and has dimensions `(band, y, x)`.
The name `rgb` makes the purpose of the variable clear to readers.
xarray determines the plotting behaviour from the dimensions and the plotting method.
The order of the three entries along `band` associates them with the red, green, and blue display channels.

Calling the general `.plot()` method on this 3D DataArray produces a different result:

In [ ]:
rgb.plot()

The result is a histogram of all values.
For the 2D single-band DataArray, xarray could use `y` and `x` as the image plane.
The `rgb` DataArray has an additional `band` dimension, so the general plotting method chooses the histogram.

Use `.plot.imshow()` when the extra dimension contains the colour channels:

In [ ]:
rgb.plot.imshow(vmin=0, vmax=3000)

Here `.plot.imshow()` interprets the length-three `band` dimension as the red, green, and blue channels and uses `y` and `x` for the image plane.

The `vmin` and `vmax` arguments set the display stretch for every channel.
Values at or below `vmin` map to zero intensity in that channel, while values at or above `vmax` map to full intensity.
This scaling changes only the displayed image, and the values stored in `rgb` remain unchanged.

A suitable stretch depends on the product.
The `s2_geomad_annual` product used here is derived from Sentinel-2 surface reflectance, for which 0 to 3000 is a commonly used display stretch.
Landsat surface reflectance commonly uses 0 to 7500.
Running the plot without `vmin` and `vmax` shows how xarray's automatic scaling changes the appearance of the composite.

## F. Facet grid

A facet grid places related plots in separate panels under a shared plotting rule.
This arrangement makes the 2024 and 2025 red-band images easier to compare side by side.

In [ ]:
ds.red.plot(col="time", cmap="Reds", vmin=0, vmax=3000)

The argument `col="time"` creates one panel for each value on the `time` coordinate, and each panel contains a 2D `(y, x)` image.
The same `cmap`, `vmin`, and `vmax` settings apply to every panel.
A given pixel value therefore appears as the same colour in each year, so colour differences between years reflect differences in the data.

Faceting can also separate the entries along the `band` dimension:

In [ ]:
rgb.plot(col="band", vmin=0, vmax=3000)

This plot contains one 2D panel for each of the red, green, and blue bands.
The panels show the three input bands separately, which makes their differences easier to inspect.

## G. Next steps

Continue to [`EXERCISE_Beginners_Guide.ipynb`](./EXERCISE_Beginners_Guide.ipynb) to apply the Open Data Cube, xarray, and plotting skills from the guide.